# LUMIERE pairs: geometry check and a 4-pair sample (read-only)

EXP-2026-005, steps 1 and 3. For the 36 same-week pairs that have a pre-op tumor mask, this notebook

- measures whether DeepBraTumIA's atlas copies are rescaled (from its transform text files) and how thick the native T1ce slices are (from file headers);
- copies the files of 4 pairs (3 complete and 1 partial resection, seed 2026) into one tar file and downloads it through the browser, so the code can be tested on real files on the laptop.

It never writes to Drive and prints counts only, without patient or study names. No GPU is needed: a CPU runtime is enough.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-lumiere"
LUMIERE_ROOT = "/content/drive/MyDrive/LUMIERE"
WORKERS = 16  # parallel folder listings on the Drive mount
INVENTORY = "/content/lumiere_inventory.json"  # reused if the inventory notebook ran in this runtime
SAMPLE_SEED = 2026
ARCHIVE = "/content/lumiere_pairs_sample_v1.tar"

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(LUMIERE_ROOT).is_dir(), f"{LUMIERE_ROOT} not found: check the folder name, or add a shortcut to it in My Drive"
if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
if importlib.util.find_spec("nibabel") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

In [ ]:
import json

from rhuh_synthesis import lumiere as L
from rhuh_synthesis import lumiere_pairs as P

if Path(INVENTORY).exists() and json.loads(Path(INVENTORY).read_text())["root"] == LUMIERE_ROOT:
    inventory = json.loads(Path(INVENTORY).read_text())
    print("inventory reused from this runtime")
else:
    inventory = L.scan(LUMIERE_ROOT, workers=WORKERS, progress=print)  # about 3 minutes
    Path(INVENTORY).write_text(json.dumps(inventory))
tables, table_check = L.resolve_public_tables(LUMIERE_ROOT, inventory["tables"], "/content")
ratings, completeness = L.read_ratings(tables["ratings"]), L.read_completeness(tables["completeness"])
pairs = P.eligible_pairs(inventory, ratings, completeness)
print(table_check)
print("eligible same-week pairs:", len(pairs))

In [ ]:
rows = P.geometry_rows(LUMIERE_ROOT, pairs, workers=8, progress=print)
Path("/content/lumiere_pair_geometry.json").write_text(json.dumps(rows))  # holds names: stays on this runtime
print("GEOMETRY SUMMARY")
print(json.dumps(P.summarize_geometry(rows, pairs), indent=1))

In [ ]:
from google.colab import files

sample = P.choose_sample(pairs, seed=SAMPLE_SEED)
if Path(ARCHIVE).exists():
    Path(ARCHIVE).unlink()
packed = P.pack_pairs(LUMIERE_ROOT, sample, ARCHIVE)
print(f"ARCHIVE pairs {len(sample)} | files {packed['files']} | MiB {packed['bytes'] / 2 ** 20:.1f} | sha256 {packed['sha256']}")
files.download(ARCHIVE)

## What to send back

1. Copy everything printed after `GEOMETRY SUMMARY`, and the line that starts with `ARCHIVE`. They hold counts only.
2. Move the downloaded file `lumiere_pairs_sample_v1.tar` into the repo folder on the laptop as `data/LUMIERE/lumiere_pairs_sample_v1.tar` (create the `LUMIERE` folder; `data/` is git-ignored). Do not unpack it.

If a cell stops with an error, send the error message instead. If the browser blocks the download, allow downloads for this page and run the last cell again.